In [1]:
import pandas as pd
import numpy as np
import panel as pn
import holoviews as hv
from holoviews import opts
from scipy import stats

pn.extension()
hv.extension('bokeh') 

In [2]:
df = pd.read_csv('data/enriched_outdoor_games.csv')

print(f"loaded {len(df)} games")
print(df.head())

loaded 732 games
           game_id  season  week     gameday   weekday home_team away_team  \
0   2020_01_HOU_KC    2020     1  2020-09-10  Thursday        KC       HOU   
1  2020_01_CLE_BAL    2020     1  2020-09-13    Sunday       BAL       CLE   
2  2020_01_NYJ_BUF    2020     1  2020-09-13    Sunday       BUF       NYJ   
3   2020_01_LV_CAR    2020     1  2020-09-13    Sunday       CAR        LV   
4   2020_01_MIA_NE    2020     1  2020-09-13    Sunday        NE       MIA   

   home_score  away_score  temp  ...  point_differential home_win    met_lat  \
0          34          20  56.0  ...                  14        1  39.048936   
1          38           6  76.0  ...                  32        1  39.277973   
2          27          17  67.0  ...                  10        1  42.773733   
3          30          34  81.0  ...                  -4        0  35.225774   
4          21          11  70.0  ...                  10        1  42.090888   

     met_lon met_temp_C  met_dewp

# cold weather = northern teams

In [3]:
cold_teams = [
    'BUF',  # Buffalo Bills
    'GB',   # Green Bay Packers
    'CHI',  # Chicago Bears
    'NE',   # New England Patriots
    'PIT',  # Pittsburgh Steelers
    'CLE',  # Cleveland Browns
    'MIN',  # Minnesota Vikings
    'DET',  # Detroit Lions
    'NYG',  # New York Giants
    'NYJ',  # New York Jets
    'PHI',  # Philadelphia Eagles
    'BAL',  # Baltimore Ravens
    'CIN',  # Cincinnati Bengals
    'DEN',  # Denver Broncos
    'KC'    # Kansas City Chiefs
]

# warm weather = southern/dome teams

In [4]:
warm_teams = [
    'MIA',  # Miami Dolphins
    'TB',   # Tampa Bay Buccaneers
    'DAL',  # Dallas Cowboys
    'ARI',  # Arizona Cardinals
    'LAR',  # Los Angeles Rams
    'LAC',  # Los Angeles Chargers
    'JAX',  # Jacksonville Jaguars
    'HOU',  # Houston Texans
    'NO',   # New Orleans Saints
    'ATL',  # Atlanta Falcons
    'CAR',  # Carolina Panthers
    'TEN',  # Tennessee Titans
    'LV'    # Las Vegas Raiders
]

In [5]:
print(f"cold teams: {len(cold_teams)}")
print(f"warm teams: {len(warm_teams)}")

cold teams: 15
warm teams: 13


In [6]:
hot_threshold = 75
cold_threshold = 40

# cold teams playing away in hot weather
cold_in_heat = df[(df['away_team'].isin(cold_teams)) & (df['temp'] > hot_threshold)]

# warm teams playing away in cold weather
warm_in_cold = df[(df['away_team'].isin(warm_teams)) & (df['temp'] < cold_threshold)]

# baselines for comparison
cold_away_all = df[df['away_team'].isin(cold_teams)]
warm_away_all = df[df['away_team'].isin(warm_teams)]

print(f"found {len(cold_in_heat)} games: cold teams in heat")
print(f"found {len(warm_in_cold)} games: warm teams in cold")

found 56 games: cold teams in heat
found 48 games: warm teams in cold


In [7]:
# calculate win rates and avg points scored

def win_rate(games_df):
    if len(games_df) == 0:
        return 0
    wins = (games_df['away_score'] > games_df['home_score']).sum()
    return wins / len(games_df) * 100

def avg_points(games_df):
    if len(games_df) == 0:
        return 0
    return games_df['away_score'].mean()

In [8]:
# cold teams stats
cold_heat_wr = win_rate(cold_in_heat)
cold_baseline_wr = win_rate(cold_away_all)
cold_heat_pts = avg_points(cold_in_heat)
cold_baseline_pts = avg_points(cold_away_all)

# warm teams stats
warm_cold_wr = win_rate(warm_in_cold)
warm_baseline_wr = win_rate(warm_away_all)
warm_cold_pts = avg_points(warm_in_cold)
warm_baseline_pts = avg_points(warm_away_all)

# differences
cold_diff_wr = cold_heat_wr - cold_baseline_wr
cold_diff_pts = cold_heat_pts - cold_baseline_pts
warm_diff_wr = warm_cold_wr - warm_baseline_wr
warm_diff_pts = warm_cold_pts - warm_baseline_pts

In [9]:
print("--- STATISTICAL SIGNIFICANCE TEST (T-Test) ---")

# 1. Test Warm Teams (the big drop)
# We compare the scores in cold games vs ALL away games
warm_cold_scores = warm_in_cold['away_score']
warm_baseline_scores = warm_away_all['away_score']

t_stat, p_val = stats.ttest_ind(warm_cold_scores, warm_baseline_scores, equal_var=False)

print(f"WARM TEAMS P-Value: {p_val:.4f}")
if p_val < 0.05:
    print(">> RESULT: Statistically Significant! (We are 95% sure this isn't random)")
else:
    print(">> RESULT: Not significant (might need more data)")

print("-" * 30)

# 2. Test Cold Teams (the slight increase)
cold_heat_scores = cold_in_heat['away_score']
cold_baseline_scores = cold_away_all['away_score']

t_stat_c, p_val_c = stats.ttest_ind(cold_heat_scores, cold_baseline_scores, equal_var=False)

print(f"COLD TEAMS P-Value: {p_val_c:.4f}")
if p_val_c < 0.05:
    print(">> RESULT: Statistically Significant!")
else:
    print(">> RESULT: Not significant (likely random noise)")

--- STATISTICAL SIGNIFICANCE TEST (T-Test) ---
WARM TEAMS P-Value: 0.6989
>> RESULT: Not significant (might need more data)
------------------------------
COLD TEAMS P-Value: 0.8791
>> RESULT: Not significant (likely random noise)


In [10]:
print("COLD TEAMS:")
print(f"  in heat: {cold_heat_wr:.1f}% wins, {cold_heat_pts:.1f} pts")
print(f"  baseline: {cold_baseline_wr:.1f}% wins, {cold_baseline_pts:.1f} pts")
print(f"  difference: {cold_diff_wr:+.1f}% wins, {cold_diff_pts:+.1f} pts")

print("\nWARM TEAMS:")
print(f"  in cold: {warm_cold_wr:.1f}% wins, {warm_cold_pts:.1f} pts")
print(f"  baseline: {warm_baseline_wr:.1f}% wins, {warm_baseline_pts:.1f} pts")
print(f"  difference: {warm_diff_wr:+.1f}% wins, {warm_diff_pts:+.1f} pts")


COLD TEAMS:
  in heat: 44.6% wins, 20.9 pts
  baseline: 42.2% wins, 20.7 pts
  difference: +2.5% wins, +0.2 pts

WARM TEAMS:
  in cold: 27.1% wins, 19.6 pts
  baseline: 38.7% wins, 20.2 pts
  difference: -11.7% wins, -0.6 pts


In [11]:
# bar chart for win rate
cold_wr_data = pd.DataFrame({
    'condition_cold': ['Extreme Heat', 'Baseline'],
    'win_rate': [cold_heat_wr, cold_baseline_wr]
})

cold_wr_chart = hv.Bars(cold_wr_data, 'condition_cold', 'win_rate').opts(
    title='Cold Teams: Win Rate',
    width=400, height=300,
    color='condition_cold', cmap=['#ef4444', '#3b82f6'],
    ylabel='Win %',
    xlabel='Condition',
    ylim=(min(cold_heat_wr, cold_baseline_wr)-5, max(cold_heat_wr, cold_baseline_wr)+5)
)

# violin plot for points distribution
violin_data = []
for score in cold_in_heat['away_score']:
    violin_data.append(('Extreme Heat', score))
for score in cold_away_all['away_score']:
    violin_data.append(('Baseline', score))

violin_df = pd.DataFrame(violin_data, columns=['condition_cold', 'points'])

cold_pts_violin = hv.Violin(violin_df, 'condition_cold', 'points').opts(
    title='Cold Teams: Points Distribution',
    width=400, height=300,
    ylabel='Points Scored',
    xlabel='Condition',
    violin_color='condition_cold',
    cmap=['#ef4444', '#3b82f6']
)

(cold_wr_chart + cold_pts_violin)

:Layout
   .Bars.I   :Bars   [condition_cold]   (win_rate)
   .Violin.I :Violin   [condition_cold]   (points)

In [12]:
warm_wr_data = pd.DataFrame({
    'condition_warm': ['Extreme Cold', 'Baseline'],
    'win_rate': [warm_cold_wr, warm_baseline_wr]
})

# scatter + curve for win rate (shows the drop clearly)
warm_wr_scatter = hv.Scatter(warm_wr_data, 'condition_warm', 'win_rate').opts(
    size=15, color='#06b6d4'
)
warm_wr_curve = hv.Curve(warm_wr_data, 'condition_warm', 'win_rate').opts(
    line_width=3, color='#06b6d4'
)
warm_wr_plot = (warm_wr_scatter * warm_wr_curve).opts(
    title='Warm Teams: Win Rate',
    width=400, height=300,
    ylabel='Win %',
    xlabel='Condition',
    ylim=(min(warm_cold_wr, warm_baseline_wr)-5, max(warm_cold_wr, warm_baseline_wr)+5)
)

# box plot for points (shows median, quartiles, outliers)
box_data = []
for score in warm_in_cold['away_score']:
    box_data.append(('Extreme Cold', score))
for score in warm_away_all['away_score']:
    box_data.append(('Baseline', score))

box_df = pd.DataFrame(box_data, columns=['condition_warm', 'points'])

warm_pts_box = hv.BoxWhisker(box_df, 'condition_warm', 'points').opts(
    title='Warm Teams: Points Distribution',
    width=400, height=300,
    ylabel='Points Scored',
    xlabel='Condition',
    box_fill_color='condition_warm',
    cmap=['#06b6d4', '#3b82f6']
)

(warm_wr_plot + warm_pts_box)

:Layout
   .Overlay.I    :Overlay
      .Scatter.I :Scatter   [condition_warm]   (win_rate)
      .Curve.I   :Curve   [condition_warm]   (win_rate)
   .BoxWhisker.I :BoxWhisker   [condition_warm]   (points)

In [13]:
diff_data = pd.DataFrame({
    'scenario': ['Cold in Heat', 'Warm in Cold'],
    'win_rate_diff': [cold_heat_wr - cold_baseline_wr, warm_cold_wr - warm_baseline_wr],
    'points_diff': [cold_heat_pts - cold_baseline_pts, warm_cold_pts - warm_baseline_pts]
})

# color: red if negative (worse), green if positive (better)
colors_wr = ['#22c55e' if x > 0 else '#ef4444' for x in diff_data['win_rate_diff']]
colors_pts = ['#22c55e' if x > 0 else '#ef4444' for x in diff_data['points_diff']]

# win rate bars with colors
wr_bars = []
for i, row in diff_data.iterrows():
    bar_data = pd.DataFrame({
        'scenario': [row['scenario']],
        'win_rate_diff': [row['win_rate_diff']]
    })
    wr_bars.append(hv.Bars(bar_data, 'scenario', 'win_rate_diff').opts(color=colors_wr[i]))

wr_diff_chart = hv.Overlay(wr_bars).opts(
    title='Win Rate: Change from Baseline',
    width=450, height=300,
    invert_axes=True,
    xlabel='Percentage Points',
    ylabel=''
)

# points bars with colors
pts_bars = []
for i, row in diff_data.iterrows():
    bar_data = pd.DataFrame({
        'scenario': [row['scenario']],
        'points_diff': [row['points_diff']]
    })
    pts_bars.append(hv.Bars(bar_data, 'scenario', 'points_diff').opts(color=colors_pts[i]))

pts_diff_chart = hv.Overlay(pts_bars).opts(
    title='Points: Change from Baseline',
    width=450, height=300,
    invert_axes=True,
    xlabel='Points per Game',
    ylabel=''
)

# add zero lines
zero_line_wr = hv.VLine(0).opts(color='black', line_width=2)
zero_line_pts = hv.VLine(0).opts(color='black', line_width=2)

((wr_diff_chart * zero_line_wr) + (pts_diff_chart * zero_line_pts))

:Layout
   .Overlay.I  :Overlay
      .Bars.I  :Bars   [scenario]   (win_rate_diff)
      .Bars.II :Bars   [scenario]   (win_rate_diff)
      .VLine.I :VLine   [x,y]
   .Overlay.II :Overlay
      .Bars.I  :Bars   [scenario]   (points_diff)
      .Bars.II :Bars   [scenario]   (points_diff)
      .VLine.I :VLine   [x,y]

In [14]:
print("Cold teams in extreme heat:")
if len(cold_in_heat) > 0:
    cold_breakdown = cold_in_heat.groupby('away_team').agg({
        'game_id': 'count',
        'away_score': 'mean'
    })
    
    # calculate wins separately
    cold_wins = cold_in_heat.groupby('away_team').apply(
        lambda x: (x['away_score'] > x['home_score']).sum(),
        include_groups=False
    )
    
    cold_breakdown.columns = ['games', 'avg_pts']
    cold_breakdown['wins'] = cold_wins
    cold_breakdown['win_pct'] = (cold_breakdown['wins'] / cold_breakdown['games'] * 100).round(1)
    print(cold_breakdown.sort_values('games', ascending=False))

print("\n\nWarm teams in extreme cold:")
if len(warm_in_cold) > 0:
    warm_breakdown = warm_in_cold.groupby('away_team').agg({
        'game_id': 'count',
        'away_score': 'mean'
    })
    
    # calculate wins separately
    warm_wins = warm_in_cold.groupby('away_team').apply(
        lambda x: (x['away_score'] > x['home_score']).sum(),
        include_groups=False
    )
    
    warm_breakdown.columns = ['games', 'avg_pts']
    warm_breakdown['wins'] = warm_wins
    warm_breakdown['win_pct'] = (warm_breakdown['wins'] / warm_breakdown['games'] * 100).round(1)
    print(warm_breakdown.sort_values('games', ascending=False))

Cold teams in extreme heat:
           games    avg_pts  wins  win_pct
away_team                                 
NYJ            8  13.375000     1     12.5
CIN            6  18.166667     1     16.7
BUF            5  32.400000     4     80.0
DEN            4  26.000000     3     75.0
CLE            4  17.000000     0      0.0
GB             4  24.750000     3     75.0
BAL            4  28.750000     3     75.0
PHI            4  21.750000     3     75.0
KC             4  30.750000     3     75.0
NYG            4  14.500000     1     25.0
MIN            3  19.666667     1     33.3
NE             3  14.666667     0      0.0
CHI            1   3.000000     0      0.0
DET            1  20.000000     1    100.0
PIT            1  13.000000     1    100.0


Warm teams in extreme cold:
           games    avg_pts  wins  win_pct
away_team                                 
MIA            7  21.000000     0      0.0
HOU            6  15.333333     2     33.3
DAL            5  27.200000     2     4

In [15]:
print("="*60)
print("FINAL RESULTS")
print("="*60)

cold_diff = cold_heat_wr - cold_baseline_wr
warm_diff = warm_cold_wr - warm_baseline_wr

print(f"\n1. COLD WEATHER TEAMS in extreme heat:")
print(f"   Win rate change: {cold_diff:+.1f}%")
if abs(cold_diff) < 3:
    print("   VERDICT: MYTH BUSTED - no real effect!")
elif cold_diff < 0:
    print(f"   VERDICT: MYTH CONFIRMED - they struggle!")
else:
    print(f"   VERDICT: MYTH BUSTED - they're actually BETTER!")

print(f"\n2. WARM WEATHER TEAMS in extreme cold:")
print(f"   Win rate change: {warm_diff:+.1f}%")
if abs(warm_diff) < 3:
    print("   VERDICT: MYTH BUSTED - no real effect!")
elif warm_diff < 0:
    print(f"   VERDICT: MYTH CONFIRMED - they struggle!")
else:
    print(f"   VERDICT: MYTH BUSTED - they're actually BETTER!")

print(f"\n{'-'*60}")
print(f"KEY FINDING: Warm teams struggle in cold (-11.7%)")
print(f"             Cold teams do fine in heat (+2.5%)")
print(f"{'-'*60}")
print(f"\nSample: {len(cold_in_heat)} games (cold/heat)")
print(f"        {len(warm_in_cold)} games (warm/cold)")
print("="*60)

FINAL RESULTS

1. COLD WEATHER TEAMS in extreme heat:
   Win rate change: +2.5%
   VERDICT: MYTH BUSTED - no real effect!

2. WARM WEATHER TEAMS in extreme cold:
   Win rate change: -11.7%
   VERDICT: MYTH CONFIRMED - they struggle!

------------------------------------------------------------
KEY FINDING: Warm teams struggle in cold (-11.7%)
             Cold teams do fine in heat (+2.5%)
------------------------------------------------------------

Sample: 56 games (cold/heat)
        48 games (warm/cold)


In [16]:
results = pd.DataFrame({
    'scenario': ['cold in heat', 'warm in cold'],
    'win_rate': [cold_heat_wr, warm_cold_wr],
    'baseline_win_rate': [cold_baseline_wr, warm_baseline_wr],
    'difference': [cold_diff, warm_diff],
    'sample_size': [len(cold_in_heat), len(warm_in_cold)]
})

print(results)
# results.to_csv('my_results.csv', index=False)

       scenario   win_rate  baseline_win_rate  difference  sample_size
0  cold in heat  44.642857          42.165242    2.477615           56
1  warm in cold  27.083333          38.745387  -11.662054           48


# 🏈 Final Conclusions

### 1. Cold Teams in Heat: MYTH BUSTED
* **Observation:** Cold teams actually had a slightly **higher win rate (+2.5%)** in extreme heat compared to their baseline.
* **Verdict:** There is no evidence that heat negatively impacts northern teams. They perform just fine.

### 2. Warm Teams in Cold: PLAUSIBLE (But Nuanced)
* **Observation:** Warm teams saw a **major drop in win rate (-11.7%)** when playing in extreme cold (below 40°F).
* **Statistical Context:** Interestingly, our T-Test on *points scored* showed no statistically significant drop ($p=0.69$).
* **Verdict:** **Myth Plausible.** While warm teams lose significantly more often in the cold, the scoring data suggests they aren't getting "blown out." They are likely losing closer games, perhaps due to factors that don't show up in the final score (e.g., turnovers, field goal accuracy, or fatigue in the 4th quarter).

In [18]:
dashboard = pn.Column(
    pn.Tabs(
        ('Cold Teams', cold_wr_chart + cold_pts_violin),
        ('Warm Teams', warm_wr_plot + warm_pts_box),
        ('Comparisons', (wr_diff_chart * zero_line_wr) + (pts_diff_chart * zero_line_pts))
    )
)

dashboard.save('nfl_weather_analysis.html')

dashboard.servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'938b74dd-4088-4444-8b85-415b81b8f754': {'version…